# 02: Full Module 1 pipeline

Steps 5 to 10: IMD gridded rainfall, SoilGrids soil, merged daily weather from all sources,
season summaries, data-quality checks and a pull log. Prices and yields (steps 7 and 8) have their
own section at the end because they need things only you can provide.

In [ ]:
# Setup: get the latest code from GitHub and install the IMD reader
!rm -rf crop-advisor && git clone -q --depth 1 https://github.com/finnmcmissile49/crop-advisor
!pip -q install imdlib
import sys, pandas as pd
sys.path.insert(0, 'crop-advisor/src')
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)

## Where data is saved

By default, data goes to the Colab machine and disappears when the runtime stops. Set
`USE_DRIVE = True` to keep it in your Google Drive (Colab will ask you to allow access).

In [ ]:
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/crop-advisor-data'
else:
    DATA_DIR = '/content/data'
START_YEAR, END_YEAR = 2021, 2023   # widen to 1997, 2023 once this works
print('Saving to', DATA_DIR)

## Run everything (steps 5, 6, 9, 10)

In [ ]:
from cropadvisor.pipeline import run
res = run(START_YEAR, END_YEAR, DATA_DIR, locations_csv='crop-advisor/data/locations.csv')
print(res['log'][['source', 'status', 'rows', 'note']].to_string(index=False))

## Step 5: three rainfall sources, kharif seasons

IMD is built from rain gauges, so it is the closest thing to ground truth here. Open-Meteo (ERA5)
and NASA POWER are model and satellite products.

Watch Pune: NASA POWER's grid cell (about 55 km wide) reaches into the Western Ghats, so it reports
2 to 4 times IMD's rain. A point 55 km east gives a value close to IMD.

In [ ]:
s = res['seasons']
k = s[s.season == 'kharif']
cols = [c for c in ['name', 'year', 'rain_total_imd', 'rain_total_om', 'rain_total_pw',
                    'rain_diff_om_imd', 'rain_diff_pw_imd', 'sources_disagree'] if c in k]
print(k[cols].to_string(index=False))

## Step 6: soil (0-100 cm, SoilGrids)

SoilGrids has no data for towns, so each location uses farmland nearby (saved in
`data/locations.csv` as `soil_lat`, `soil_lon`; `soil_distance_km` is how far away it is).

In [ ]:
soil = res['soil']
cols = [c for c in ['texture', 'clay_pct', 'sand_pct', 'silt_pct', 'ph', 'soc_g_kg',
                    'bulk_density_g_cm3', 'field_capacity', 'wilting_point', 'awc_mm_per_m',
                    'clay_above_model_range', 'soil_distance_km', 'error'] if c in soil]
print(soil[cols].to_string())

## Step 9: season features for Module 2

In [ ]:
cols = ['name', 'year', 'season', 'rain_total_om', 'rainy_days_om', 'longest_dry_spell_om',
        'tmax_mean', 'hot_days_35c', 'et0_total_om', 'water_balance_om']
print(s[cols].to_string(index=False))

## Step 9: data-quality report

In [ ]:
qc = res['qc']
print('No issues found' if qc.empty else qc.to_string(index=False))

## Steps 7 and 8: prices and yields

**Prices** need a free data.gov.in API key. Register at data.gov.in, then add the key as a Colab
secret named `DATA_GOV_IN_KEY` (key icon in the left sidebar) and turn on notebook access.
This cell skips itself until the key exists.

**Yields** have no public API. Download district crop statistics for Maharashtra from the DES /
UPAg portal as CSV, upload it here, and map its columns with `cropadvisor.yields.standardise`.

In [ ]:
try:
    from google.colab import userdata
    key = userdata.get('DATA_GOV_IN_KEY')
except Exception:
    key = None
if key:
    from cropadvisor.prices import fetch_prices, COMMODITIES
    p = fetch_prices(key, COMMODITIES['soybean'])
    print(len(p), 'price rows'); print(p.head().to_string())
else:
    print('No DATA_GOV_IN_KEY secret yet: skipping prices.')